# Task 2 -- Unsupervised Domain Adaptation on PACS

**Before running:** Settings (right sidebar) -> Accelerator: **GPU T4 x2**, Internet: **On**.

**Inputs you need attached** (Add Input, right sidebar):
1. Your `task2_pa1.zip` code, uploaded as a Kaggle dataset (same way you did `task1_pa1.zip`).
2. A PACS dataset, e.g. search "PACS" in Add Input and add `ma3ple/pacs-dataset` or `nickfratto/pacs-dataset`.

Every path below is auto-discovered with `glob` -- you do not need to know or edit any `/kaggle/input/...` path yourself.

In [ ]:
!nvidia-smi
!curl -sI https://www.google.com | head -1

In [ ]:
# --- Locate this task's code: handle both a still-zipped upload and
#     Kaggle's auto-extract-on-upload behavior, and copy to /kaggle/working
#     since /kaggle/input is read-only ---
import glob, os, zipfile, shutil

hits = glob.glob("/kaggle/input/**/evaluate_final.py", recursive=True)
if hits:
    src_root_dir = os.path.dirname(os.path.dirname(hits[0]))
    print("Found already-extracted code at:", src_root_dir)
    os.makedirs("/kaggle/working/pa2", exist_ok=True)
    shutil.copytree(src_root_dir, "/kaggle/working/pa2", dirs_exist_ok=True)
else:
    candidates = [p for p in glob.glob("/kaggle/input/**/*.zip", recursive=True)
                  if "task2" in os.path.basename(p).lower() or "pa2" in os.path.basename(p).lower()]
    assert candidates, "Could not find task2/ (extracted) or task2_pa1.zip under /kaggle/input."
    code_zip = candidates[0]
    print("Using code zip:", code_zip)
    os.makedirs("/kaggle/working/pa2", exist_ok=True)
    with zipfile.ZipFile(code_zip) as zf:
        zf.extractall("/kaggle/working/pa2")

hits2 = glob.glob("/kaggle/working/pa2/**/evaluate_final.py", recursive=True)
assert hits2, "task2/evaluate_final.py not found after copy/extract -- check input contents."
task2_dir = os.path.dirname(hits2[0])
root_dir = os.path.dirname(task2_dir)
print("task2/ ready at:", task2_dir)
assert os.path.isdir(os.path.join(root_dir, "shared")), "shared/ not found next to task2/."

%cd {task2_dir}
print("Working directory:", os.getcwd())


In [ ]:
# --- Locate PACS ---
import glob, os
from collections import defaultdict

candidates = defaultdict(int)
for name in ["photo", "art_painting", "cartoon", "sketch"]:
    seen_for_this_name = set()
    for p in glob.glob(f"/kaggle/input/**/{name}", recursive=True):
        seen_for_this_name.add(os.path.dirname(p))
    for p in glob.glob(f"/kaggle/input/**/{name.replace('_', ' ')}", recursive=True):
        seen_for_this_name.add(os.path.dirname(p))
    for root in seen_for_this_name:
        candidates[root] += 1

print("All candidate PACS roots found (how many of the 4 domains matched):")
for root, count in candidates.items():
    print(f"  {count}/4  {root}")

# a valid root needs all 4 domains, and must not be a derived/transformed
# variant (e.g. "dct" = DCT/frequency-transformed images, not real photos --
# some public PACS mirrors bundle both alongside each other)
valid = [r for r, c in candidates.items() if c == 4 and "dct" not in r.lower()]
assert valid, f"No clean 4/4 PACS root found. All candidates: {dict(candidates)}"
if len(valid) > 1:
    print("WARNING: multiple valid candidates, picking the first -- verify this is right:", valid)
pacs_root = valid[0]
print("\nSelected PACS root:", pacs_root)


In [ ]:
# --- Point config.yaml at the discovered PACS root ---
import yaml

with open("configs/config.yaml") as f:
    cfg = yaml.safe_load(f)
cfg["dataset"]["root"] = pacs_root
with open("configs/config.yaml", "w") as f:
    yaml.safe_dump(cfg, f, sort_keys=False)
print("dataset.root set to:", cfg["dataset"]["root"])

In [ ]:
# --- Smoke test (seconds, no real data needed) -- run this before anything else ---
!python tests/test_pipeline.py

If the smoke test above shows any `[FAIL]` lines, stop here and fix/report them before training -- everything below burns real GPU time.

In [ ]:
!python train.py --method source_only

In [ ]:
!python train.py --method dan

In [ ]:
!python train.py --method dann

In [ ]:
!python train.py --method cdan

### Controlled design study: DAN's lambda_MMD sweep
`lambda_mmd=1.0` is already covered by the main DAN run above, so only the two extra points need training.

In [ ]:
!python train.py --method dan --lambda_mmd 0.1  --tag _lam0.1

In [ ]:
!python train.py --method dan --lambda_mmd 10.0 --tag _lam10.0

In [ ]:
!python evaluate_final.py

In [ ]:
# --- Zip everything you need to download ---
import shutil
shutil.make_archive("/kaggle/working/task2_results", "zip", ".", "results")
shutil.make_archive("/kaggle/working/task2_checkpoints", "zip", ".", "checkpoints")
print("Download task2_results.zip and task2_checkpoints.zip from the Output pane on the right.")